# 01 — Market data & visualisation (demo)

**Synthetic data only.** Tables are shaped like public or licensed European wholesale feeds (EPEX-style results, ENTSO-E Transparency–style actuals, NWP-style weather, TTF-style gas, EU ETS–style CO₂) but **are not** real vendor or exchange data.

See `modules/forecasting/features/01-market-data-and-visualisation.md`.


In [ ]:
import os
import sys
from pathlib import Path

# Uploaded demo modules (Databricks). `dbfs:/tmp/...` is readable on shared UC clusters; FileStore path is a fallback.
sys.path.insert(0, "/dbfs/tmp/energy-trading-forecast-demo")
sys.path.insert(0, "/dbfs/FileStore/energy-trading-forecasting-demo/demo_data")
_env = os.environ.get("DEMO_DATA_PATH", "").strip()
if _env:
    sys.path.insert(0, _env)

CWD = Path.cwd()
for dd in (
    CWD / "modules" / "forecasting" / "demo_data",
    CWD / "demo_data",
    CWD.parent / "demo_data",
):
    if (dd / "notebook_helpers.py").is_file():
        sys.path.insert(0, str(dd))
        break

import notebook_helpers as nh
nh.ensure_demo_data_on_path()
# Matches modules/forecasting/README.md → unity_catalog.schemas (override with DEMO_UC_* env)
print(
    "Unity Catalog Delta target:",
    nh.catalog_schema(),
    "— example:",
    nh.full_table_name("demo_prices_spot_hourly"),
)

import synthetic_generators as sg
import european_demo_sources as eds

spark = nh.get_spark()
cat, sch = nh.catalog_schema()
print(f"Target catalog: {cat}.{sch} | Spark session: {spark is not None}")
print("Source families:", list(eds.SIMULATED_SOURCE_FAMILIES.keys()))


In [ ]:

# Reference data & prices (hourly, ~90d)
nh.write_demo_table(
    "demo_reference_bidding_zones",
    sg.bidding_zones_rows(),
    ("zone_code", "name", "country", "eic_bidding_zone"),
    spark=spark,
)
nh.write_demo_table(
    "demo_prices_spot_hourly",
    sg.prices_spot_rows(),
    ("delivery_ts", "bidding_zone", "product_code", "venue", "price_eur_mwh", "unit"),
    spark=spark,
)
nh.write_demo_table(
    "demo_bronze_prices_spot",
    eds.bronze_prices_spot_rows(),
    eds.BRONZE_PRICES_SPOT_COLUMNS,
    spark=spark,
)
nh.write_demo_table(
    "demo_entsoe_transparency_style",
    eds.entsoe_transparency_style_rows(72 * 24),
    eds.ENTSOE_STYLE_COLUMNS,
    spark=spark,
)
nh.write_demo_table(
    "demo_german_imbalance_prices_style",
    eds.german_imbalance_prices_style_rows(168),
    eds.GERMAN_IMBALANCE_COLUMNS,
    spark=spark,
)
nh.write_demo_table(
    "demo_nwp_ecmwf_surface_style",
    eds.nwp_ecmwf_style_rows(168),
    eds.NWP_ECMWF_STYLE_COLUMNS,
    spark=spark,
)
nh.write_demo_table(
    "demo_gas_hub_ttf_style",
    eds.gas_hub_ttf_style_rows(90),
    eds.GAS_TTF_STYLE_COLUMNS,
    spark=spark,
)
nh.write_demo_table(
    "demo_eua_ets_daily_style",
    eds.eua_ets_daily_rows(90),
    eds.EUA_ETS_COLUMNS,
    spark=spark,
)
nh.write_demo_table(
    "demo_grid_signals",
    sg.grid_signals_rows(),
    ("ts", "tso", "metric", "unit", "value"),
    spark=spark,
)
nh.write_demo_table(
    "demo_fundamentals_de_lu",
    sg.fundamentals_rows(),
    ("ts", "zone", "temp_c", "wind_ms", "solar_mw_est", "load_mw", "residual_mw"),
    spark=spark,
)
nh.write_demo_table(
    "demo_cross_border_flows",
    sg.cross_border_flows_rows(),
    ("ts", "from_country", "to_country", "direction", "mw"),
    spark=spark,
)
print("Market data demo tables written.")
